# Stage 3 — Data Preprocessing & Feature Engineering

**Project:** Customer Churn Prediction + Explainable AI  
**Stage:** 3 — Cleaning, Feature Engineering & Pipeline Serialization  

### Objectives for Stage 3:
1. **Load raw data** safely from `data/telco_customer_churn.csv` without modifying the original file.
2. **Clean data types**, specifically handling `TotalCharges` string conversion and 11 blank whitespace values.
3. **Exclude identifiers (`customerID`)** to prevent severe model overfitting.
4. **Encode target variable (`Churn`)** into binary integers (`No` -> `0`, `Yes` -> `1`).
5. **Perform a Stratified Train/Test Split (80/20)** *before* fitting transformers to eliminate data leakage.
6. **Engineer domain features** (`AverageMonthlySpend`, `ServiceCount`) with zero leakage risk.
7. **Build a reproducible `ColumnTransformer`** with numerical imputation and One-Hot Encoding (`handle_unknown='ignore'`).
8. **Fit preprocessor strictly on training data** and transform train/test sets.
9. **Serialize preprocessor** into `models/preprocessor.joblib` for API production deployment.

## Part 1 & 2: Load Raw Dataset & Data Quality Inspection

We load the raw CSV and perform comprehensive quality checks.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Load raw dataset
raw_csv_path = '../data/telco_customer_churn.csv'
df = pd.read_csv(raw_csv_path)
print(f'Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())

Raw Dataset Shape: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print('=== Data Types & Null Counts ===')
print(df.info())

print('\n=== Missing Values (NaN) ===')
print(df.isnull().sum())

print('\n=== Duplicate Check ===')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique customerIDs: {df["customerID"].nunique()}')

=== Data Types & Null Counts ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  Paperless

## Part 3: Handling `TotalCharges`

`TotalCharges` is stored as an `object` data type in the raw CSV because 11 rows contain blank space strings (`" "`).

In [3]:
# Identify blank strings in TotalCharges
blank_mask = df['TotalCharges'].astype(str).str.strip() == ''
print(f'Number of blank space rows in TotalCharges: {blank_mask.sum()}')

# Inspect the affected records
display(df[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']])

Number of blank space rows in TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


### TotalCharges Treatment Strategy:
1. **Root Cause:** All 11 customers have `tenure == 0` months, representing brand new subscriptions.
2. **Action:** Coerce blank strings to `NaN` using `pd.to_numeric(df['TotalCharges'], errors='coerce')`.
3. **Imputation:** Rather than dropping these rows (which discards valid customer records), `NaN` values will be median-imputed via `SimpleImputer(strategy='median')` during numerical preprocessing.

In [4]:
# Convert TotalCharges to float64
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'TotalCharges converted. Null count now: {df["TotalCharges"].isnull().sum()}')
print(f'New TotalCharges d-type: {df["TotalCharges"].dtype}')

TotalCharges converted. Null count now: 11
New TotalCharges d-type: float64


## Part 4: Removing Customer Identifier (`customerID`)

`customerID` is a 100% unique alphanumeric string (`7,043` unique values across `7,043` rows).

**Why Exclude Identifier Columns from Machine Learning Features?**
- Identifiers contain arbitrary high-cardinality strings that have zero causal relationship with customer retention.
- Including them causes decision trees to split on arbitrary character patterns, leading to **severe overfitting** and failed generalization on new customer data.

## Part 5: Separate Features (`X`) and Target (`y`)

We separate predictive features `X` from target variable `y`, converting `Churn` strings (`No` / `Yes`) to binary integers (`0` / `1`).

In [5]:
X = df.drop(columns=['customerID', 'Churn']).copy()
y = df['Churn'].map({'No': 0, 'Yes': 1}).astype(int)

print('Features matrix X shape:', X.shape)
print('Target vector y shape:', y.shape)
print('\nTarget Class Distribution:')
print(y.value_counts(normalize=True).round(4) * 100)

Features matrix X shape: (7043, 19)
Target vector y shape: (7043,)

Target Class Distribution:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


## Part 6: Stratified Train/Test Split

We perform an 80/20 train/test split using `stratify=y` and `random_state=42`.

### ⚠️ Data Leakage Prevention:
We split `X` and `y` **BEFORE** fitting any feature engineering or preprocessing transformations. Fitting imputer statistics or encodings on the entire dataset prior to splitting would leak test set distribution info into training data, invalidating model evaluation.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training Set: {X_train.shape[0]} rows ({X_train.shape[0]/len(df):.1%})')
print(f'Testing Set:  {X_test.shape[0]} rows ({X_test.shape[0]/len(df):.1%})')
print('\nTraining Churn Ratio:', y_train.mean().round(4))
print('Testing Churn Ratio: ', y_test.mean().round(4))

Training Set: 5634 rows (80.0%)
Testing Set:  1409 rows (20.0%)

Training Churn Ratio: 0.2654
Testing Churn Ratio:  0.2654


## Part 11: Feature Engineering

We engineer two domain-specific features with clear business rationale:

1. `AverageMonthlySpend` = `TotalCharges / tenure` (with zero-division protection fallback to `MonthlyCharges`).
   - *Rationale:* Highlights price sensitivity when current monthly charges exceed historical average spend.
2. `ServiceCount` = Count of active subscriptions out of 9 optional services.
   - *Rationale:* Measures ecosystem stickiness. Higher service adoption increases customer switching costs.

In [7]:
def engineer_features(data):
    df_eng = data.copy()
    
    # 1. AverageMonthlySpend with zero-tenure protection
    # For tenure == 0, TotalCharges is NaN/0, so we fallback to MonthlyCharges
    df_eng['AverageMonthlySpend'] = np.where(
        df_eng['tenure'] > 0, 
        df_eng['TotalCharges'] / df_eng['tenure'], 
        df_eng['MonthlyCharges']
    )
    
    # 2. ServiceCount across 9 service columns
    services = [
        'PhoneService', 'MultipleLines', 'InternetService', 
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    
    service_count = pd.Series(0, index=df_eng.index)
    for col in services:
        # Active if value is not 'No' and not 'No phone service' and not 'No internet service'
        active_mask = ~df_eng[col].isin(['No', 'No phone service', 'No internet service'])
        service_count += active_mask.astype(int)
        
    df_eng['ServiceCount'] = service_count
    return df_eng

# Apply feature engineering to train and test sets separately
X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)

print('Engineered Training Set shape:', X_train_eng.shape)
print('Engineered Testing Set shape: ', X_test_eng.shape)
display(X_train_eng[['tenure', 'MonthlyCharges', 'TotalCharges', 'AverageMonthlySpend', 'ServiceCount']].head())

Engineered Training Set shape: (5634, 21)
Engineered Testing Set shape:  (1409, 21)


,tenure,MonthlyCharges,TotalCharges,AverageMonthlySpend,ServiceCount
3738,35,49.20,1701.65,48.618571,4
3151,15,75.10,1151.55,76.770000,3
4860,13,40.55,590.35,45.411538,4
3867,26,73.50,1905.70,73.296154,6
3810,1,44.55,44.55,44.550000,2


## Part 7, 8, 9 & 10: Building the `ColumnTransformer` Pipeline

We group engineered features into numerical and categorical categories, building modular scikit-learn pipelines.

- **Numerical Features:** `SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`, `AverageMonthlySpend`, `ServiceCount`
  - *Strategy:* `SimpleImputer(strategy='median')`.
  - *Note on Scaling:* Scaling (e.g. `StandardScaler`) is omitted because our target model family is tree-based (XGBoost/Random Forest), which is scale-invariant.
- **Categorical Features:** All remaining 15 string features.
  - *Strategy:* `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`.
  - *Safety:* `handle_unknown='ignore'` prevents API failure if unseen categories appear during inference.

In [8]:
# Identify numerical and categorical column names
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AverageMonthlySpend', 'ServiceCount']
cat_cols = [col for col in X_train_eng.columns if col not in num_cols]

print(f'Numerical features ({len(num_cols)}):', num_cols)
print(f'Categorical features ({len(cat_cols)}):', cat_cols)

# Construct numerical pipeline
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Construct categorical pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

Numerical features (6): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AverageMonthlySpend', 'ServiceCount']
Categorical features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Part 12 & 13: Fit Preprocessor strictly on Training Data & Inspect

We fit the `preprocessor` exclusively on `X_train_eng`, and then transform both `X_train_eng` and `X_test_eng`.

In [9]:
# Fit ONLY on training data
preprocessor.fit(X_train_eng)

# Transform train and test data
X_train_processed = preprocessor.transform(X_train_eng)
X_test_processed = preprocessor.transform(X_test_eng)

# Retrieve transformed feature names
feature_names = preprocessor.get_feature_names_out()

print('=== Preprocessing Output Summary ===')
print(f'X_train_processed shape: {X_train_processed.shape}')
print(f'X_test_processed shape:  {X_test_processed.shape}')
print(f'Total Transformed Features: {len(feature_names)}')

# Convert to DataFrame for visualization
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
display(X_train_df.head())

=== Preprocessing Output Summary ===
X_train_processed shape: (5634, 47)
X_test_processed shape:  (1409, 47)
Total Transformed Features: 47


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__AverageMonthlySpend,num__ServiceCount,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,...,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
0,0.0,35.0,49.20,1701.65,48.618571,4.0,0.0,1.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,15.0,75.10,1151.55,76.770000,3.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,13.0,40.55,590.35,45.411538,4.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.0,26.0,73.50,1905.70,73.296154,6.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.0,1.0,44.55,44.55,44.550000,2.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


## Part 14: Save Serialized Preprocessing Pipeline

We serialize the fitted `preprocessor` object into `models/preprocessor.joblib` so that our future FastAPI backend can preprocess incoming raw customer payloads identically.

In [10]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/preprocessor.joblib'
joblib.dump(preprocessor, model_path)
print(f'Fitted preprocessor successfully saved to: {model_path}')
print(f'File size: {os.path.getsize(model_path)} bytes')

Fitted preprocessor successfully saved to: ../models/preprocessor.joblib
File size: 8426 bytes


## Part 15: Data Quality & Integrity Verification

We run explicit assertions to confirm pipeline correctness.

In [11]:
print('=== Data Integrity Assertions ===')
# 1. No NaNs in transformed output
assert not np.isnan(X_train_processed).any(), 'NaNs detected in X_train_processed!'
assert not np.isnan(X_test_processed).any(), 'NaNs detected in X_test_processed!'
print('✓ Assertion Passed: 0 NaNs remain in transformed datasets.')

# 2. Target contains only 0 and 1
assert set(np.unique(y_train)) == {0, 1}, 'y_train target values invalid!'
assert set(np.unique(y_test)) == {0, 1}, 'y_test target values invalid!'
print('✓ Assertion Passed: Target y contains strictly binary 0 and 1 values.')

# 3. Check saved preprocessor reload
reloaded_preprocessor = joblib.load(model_path)
test_transform = reloaded_preprocessor.transform(X_test_eng)
assert np.array_equal(X_test_processed, test_transform), 'Reloaded preprocessor output mismatch!'
print('✓ Assertion Passed: Serialized preprocessor reloads and transforms identically.')

# 4. Confirm raw CSV untouched
raw_check = pd.read_csv(raw_csv_path)
assert raw_check.shape == (7043, 21), 'Raw CSV shape mutated!'
print('✓ Assertion Passed: Raw dataset CSV remains untouched.')

=== Data Integrity Assertions ===
✓ Assertion Passed: 0 NaNs remain in transformed datasets.
✓ Assertion Passed: Target y contains strictly binary 0 and 1 values.


✓ Assertion Passed: Serialized preprocessor reloads and transforms identically.


✓ Assertion Passed: Raw dataset CSV remains untouched.


## Stage 3 Summary

1. **Files Created / Updated:**
   - `notebooks/02_preprocessing.ipynb` — Complete preprocessing notebook.
   - `data/preprocessing_notes.md` — Detailed technical preprocessing documentation.
   - `models/preprocessor.joblib` — Serialized fitted `ColumnTransformer` object.
2. **Preprocessing Decisions:**
   - Coerced 11 blank space strings in `TotalCharges` to `NaN` and median-imputed them.
   - Removed `customerID` from `X` to eliminate identifier overfitting.
   - One-hot encoded 15 categorical features with `handle_unknown='ignore'`.
3. **Feature Engineering Decisions:**
   - Derived `AverageMonthlySpend` = `TotalCharges / tenure` (with zero-division fallback).
   - Derived `ServiceCount` = Count of active subscriptions out of 9 optional services.
4. **Train/Test Dimensions:**
   - Original Train: `(5634, 20)` $\rightarrow$ Transformed Train: `(5634, 47)`
   - Original Test: `(1409, 20)` $\rightarrow$ Transformed Test: `(1409, 47)`
5. **Final Number of Features:** `47` encoded features.
6. **Data Leakage Confirmation:** Transformer fitted **ONLY on `X_train`** after an 80/20 stratified split.
7. **ML Model Training Confirmation:** **0** machine learning models were trained during Stage 3.